# 01 · Customer 360 + MDM — Lumora Bank (fictional)
Three source systems describe the same customers inconsistently. This notebook standardizes them, resolves identities
(deterministic + fuzzy), applies survivorship rules and produces one **golden record** per customer with full lineage.
All data is synthetic.

In [ ]:
# ---- CONFIG ----
DATA_DIR  = "../data/sample"        # local folder, or an S3 prefix e.g. "s3://<bucket>/<prefix>/data"
PUBLISH   = False                   # True = write tables to the Glue Data Catalog (Iceberg via Athena)
DATABASE  = "<glue_database>"
S3_PATH   = "s3://<bucket>/<prefix>"  # no trailing slash
WORKGROUP = None                    # SageMaker Unified Studio: set to the project's Athena workgroup
REGION    = "us-east-1"
MODEL_ID  = "us.amazon.nova-pro-v1:0"   # any Bedrock text model enabled in REGION
# ----------------

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath("../src"))          # repo layout; in SMUS upload src/ next to notebooks/
import pandas as pd
from lumora import c360, fraud, genai, aws_io
pd.set_option("display.max_colwidth", 60); pd.set_option("display.width", 200)
rd = lambda n: aws_io.read_csv(f"{DATA_DIR}/{n}.csv")

## 1 · Bronze — the raw extracts

In [ ]:
t24, crm, cards = rd("core_banking_customers"), rd("crm_contacts"), rd("cards_accounts")
print(f"Core banking {len(t24)} | CRM {len(crm)} | Cards {len(cards)} | total {len(t24)+len(crm)+len(cards)} records")
display(t24.head(3)); display(crm.head(3)); display(cards.head(3))

## 2 · Silver → Gold — standardize, resolve, survive

In [ ]:
res = c360.run(t24, crm, cards)
silver, gold, xref = res["silver"], res["gold"], res["xref"]
print("Match links by rule:", res["links"])
print(f"{len(silver)} source records  →  {gold.customer_id.nunique()} golden customers")
display(silver[["source","name_en","name_key","mobile","mobile_std","dob_raw","dob"]].sample(6, random_state=7))

## 3 · Accuracy vs. ground truth (synthetic data only)

In [ ]:
truth = rd("ground_truth").astype({"person_id": int})
c360.evaluate(silver, truth)

## 4 · Before / after for one customer

In [ ]:
print(gold.source_system_count.value_counts().sort_index().rename("customers by # systems"))
hero = gold[gold.source_system_count == 3].sort_values("relationship_value", ascending=False).iloc[0].customer_id
print("BEFORE — one customer, three systems:"); display(xref[xref.customer_id == hero])
print("AFTER — one golden record:");             display(gold[gold.customer_id == hero].T)

## 5 · Cross-sell signal: high-balance customers with no card

In [ ]:
display(gold[(~gold.source_systems.str.contains("CARDS")) & (gold.casa_balance > 200000)]
        .sort_values("casa_balance", ascending=False)[["customer_id","name_en","segment","casa_balance"]].head(10))

## 6 · Publish (optional)

In [ ]:
os.makedirs("../output", exist_ok=True)
gold.to_csv("../output/gold_customer_360.csv", index=False); xref.to_csv("../output/gold_customer_xref.csv", index=False)
if PUBLISH:
    aws_io.publish({"silver_customer_records": silver, "gold_customer_360": gold, "gold_customer_xref": xref},
                   DATABASE, S3_PATH, WORKGROUP)

## 7 · GenAI — bilingual relationship briefing (Amazon Bedrock)

In [ ]:
for _, row in gold[gold.source_system_count == 3].sort_values("relationship_value", ascending=False).head(2).iterrows():
    profile = {k: (None if pd.isna(v) else v) for k, v in row.items()}
    print("=" * 90, f"\n{row.customer_id} | {row.name_en} | {row.name_ar}\n")
    print(genai.customer_briefing(profile, model_id=MODEL_ID, region=REGION))